# Entrega 1 — desenho e configuração científica
**Pergunta:** a informação geotemporal pode ajudar a estimar efeitos de mídia com mais precisão e estabilidade que o agregado nacional?

Este notebook prepara a investigação. Não estima posterior nem retorno. A EDA revelou população relevante, dinâmica local e canais com suportes distintos; isso motiva hipóteses, não comprova causalidade. O guia canônico em `docs/01_DESENHO_CIENTIFICO_DIDATICO.md` contém o protocolo completo.

A execução registrada é **local**. Uma sessão real do Google Colab permanece `NOT_TESTED_IN_HOSTED_COLAB`. CPU é suficiente para esta preparação; nenhuma credencial ou Google Drive é necessário.

## 1. Preparar o ambiente
No Colab novo, a célula seguinte clona exclusivamente o repositório desta entrega e instala suas dependências fixadas. Para reproduzir uma versão exata, preencha `REVISION` com o SHA publicado; `main` acompanha a versão atual e seu commit será impresso. Se a instalação pedir reinício, reinicie e execute novamente. Fora do Colab, instale `requirements.txt` no ambiente antes de iniciar este notebook.

In [1]:
from pathlib import Path
import os, sys, subprocess, importlib.util
REPOSITORY = "https://github.com/thiagossilva17/MMM_Google_Meridian_Primeira_Entrega.git"
REVISION = "main"  # Para reprodução histórica, substituir pelo SHA informado no README/commit.
in_colab = importlib.util.find_spec("google.colab") is not None
if in_colab:
    root = Path("/content/MMM_Google_Meridian_Primeira_Entrega")
    if not root.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(root)], check=True)
    remote = subprocess.check_output(["git", "-C", str(root), "remote", "get-url", "origin"], text=True).strip()
    if remote != REPOSITORY:
        raise RuntimeError("Remoto diferente do repositório autorizado")
    subprocess.run(["git", "-C", str(root), "checkout", REVISION], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(root/"requirements.txt")], check=True)
else:
    root = Path.cwd()
    if root.name == "notebooks": root = root.parent
    if not (root/"src/data_source.py").exists():
        raise RuntimeError("Abra o notebook a partir da raiz do repositório ou de notebooks/")
os.chdir(root)
if str(root) not in sys.path: sys.path.insert(0,str(root))
print("Diretório:", root)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Ambiente:", "sessão Colab — registrar evidências ao finalizar" if in_colab else "execução local")

Diretório: /workspace/scratch/39ded6201e7d/MMM_Google_Meridian_Primeira_Entrega
Commit: e1965a511bd4c18849eec772277080c996ac63f3
Ambiente: execução local


## 2. Recuperar uma fonte que possa ser conferida
A base é sintética: 40 mercados × 156 semanas. A assinatura identifica o arquivo; o contrato identifica suas propriedades. A única transformação de colunas é remover o índice exportado após verificar sua sequência. Dados brutos permanecem intactos. Não arredondamos conversões nem reinterpretamos valores negativos dos índices.

In [2]:
import json, pandas as pd, numpy as np
from src.data_source import retrieve, check_provenance, verify_bytes
from src.data_contract import load_panel, audit, eda_comparison
path = retrieve()
provenance = check_provenance()
print("SHA-256:", verify_bytes(path.read_bytes()))
print("Origem:", provenance["url"])
df = load_panel(path)
print("Dimensão analítica:", df.shape)
display(audit(df))
display(eda_comparison(df))

,variable,minimum,maximum,zeros,negatives,total,status
0,Channel0_impression,0.000000,5.192032e+06,755,0,5.523420e+09,PASS
1,Channel1_impression,0.000000,4.397540e+06,1742,0,3.248579e+09,PASS
2,Channel2_impression,0.000000,5.610156e+06,4010,0,1.625551e+09,PASS
3,Channel3_impression,0.000000,7.635147e+06,169,0,1.127450e+10,PASS
4,Channel4_impression,0.000000,6.975542e+06,820,0,6.125671e+09,PASS
5,Channel0_spend,0.000000,3.807173e+04,755,0,4.050171e+07,PASS
6,Channel1_spend,0.000000,4.239770e+04,1742,0,3.132030e+07,PASS
7,Channel2_spend,0.000000,4.168864e+04,4010,0,1.207935e+07,PASS
8,Channel3_spend,0.000000,5.949948e+04,169,0,8.786036e+07,PASS
9,Channel4_spend,0.000000,5.435281e+04,820,0,4.773068e+07,PASS


,metric,observed,eda_rounded,tolerance,status
0,Channel2_zero_fraction,0.642628,0.643,0.001,PASS
1,Channel3_zero_fraction,0.027083,0.027,0.001,PASS
2,Promo_zero_fraction,0.468269,0.470,0.010,PASS
3,Channel2_spend_share,0.055033,0.055,0.002,PASS
4,Channel3_spend_share,0.400289,0.400,0.010,PASS


SHA-256: d9ee016f7cd21f5c90b50da10a794af91a372b69f881d3caa266edd41fdafbf6
Origem: https://raw.githubusercontent.com/google/meridian/02111531f8661373aa7b6ba31c316c67d72d1dd2/meridian/data/simulated_data/csv/geo_all_channels.csv
Dimensão analítica: (6240, 19)


**Interpretação:** PASS verifica consistência estrutural e os indicadores comparados à EDA. Não prova identificação causal nem unidades monetárias. O CSV tem 20 colunas físicas e 19 após remover o índice. `revenue_per_conversion` é preservada, mas não ativa a interpretação monetária do candidato principal.

## 3. Hipóteses e estimandos
Estimaremos futuramente Δ = soma da expectativa sob mídia observada menos expectativa sob intervenção de referência, em cada draw posterior. O contraste inicial zera o canal no horizonte definido, mantendo os demais componentes. A unidade principal será conversões; Δ/gasto será eficiência em conversões por unidade de gasto, até validar receita e gasto em unidade comum. mROI é o retorno marginal de uma pequena mudança, diferente do retorno médio.

Perguntas: os dados atualizam as priors? Channel2 tem suporte? Channel3 se separa do tempo? Há heterogeneidade estimável? GEO melhora resultados comparáveis? Priors e knots alteram conclusões? Nenhuma resposta favorável é imposta. Os extremos da EDA continuam preservados.

## 4. Relações causais hipotéticas
DAG significa grafo direcionado acíclico. O esquema não foi descoberto pela correlação. População e tempo influenciam escala/demanda; demanda pode determinar tanto investimento quanto conversões. Os índices podem ser proxies imperfeitos, deixando confundimento residual. Alternativamente, sentimento pode ser mediador; ajustá-lo mudaria o efeito estimado.

```mermaid
flowchart TD
  GT["GEO e tempo"] --> D["Demanda latente"]
  P["População"] --> D
  D --> B["Investimento"]
  B --> M["Exposição"]
  M --> Y["Conversões"]
  D --> Y
  D --> C["Índices observados"]
  D --> R["Promo e orgânico"]
  R --> Y
```

**Leitura textual equivalente:** demanda → investimento → exposição → conversões, com demanda → conversões constituindo confundimento; índices observam apenas parte da demanda. Esta descrição continua legível se o Colab não renderizar Mermaid.

Ausência de confundimento é não demonstrada; não interferência e estabilidade são assumidas; suporte e medição são parcialmente avaliáveis. O registro completo de pressupostos está no capítulo 4 do guia. Promo é contínua, baseline zero hipotético; população é escala, não publicidade incremental.

## 5. Entradas e especificação reais, sem treino
O KPI entra como `non_revenue` na versão 2.1.0. Cinco pares impressão/gasto são alinhados por nome, além de orgânico, dois controles e Promo. Índices não recebem divisão por população. Não fornecemos dados por mil habitantes da EDA, pois o Meridian realiza transformações próprias.

Adstock representa memória; Hill representa saturação. O candidato usa lag máximo 8, decaimento geométrico e Hill depois de adstock. Slope default é fixo em 1. Knots controlam tempo: `None` significa 156 knots aqui, mas um no nacional. AKS (seleção automática) está desativado. A configuração executável é verificável, mas ainda requer revisão científica.

In [3]:
import yaml
from src.meridian_setup import build_input, build_spec, inspect_input, inspect_priors
config = yaml.safe_load((root/"configs/baseline_geo.yaml").read_text())
display(pd.DataFrame(config["parameters"]).T)
input_data = build_input(df)
tensor_report = inspect_input(input_data, df)
display(pd.DataFrame({k:v for k,v in tensor_report.items() if isinstance(v,dict)}).T)
model_spec = build_spec(input_data)
print("ModelSpec construído:", type(model_spec).__name__)
print("Tipo de prior paga:", model_spec.media_prior_type)

,value,status,reason,source
media_effects_dist,log_normal,verified_default,Efeitos de mídia positivos; hipótese a investi...,https://github.com/google/meridian/blob/cb0be8...
media_prior_type,contribution,explicit_choice,"Evita unidade monetária não validada; Beta(1,9...",https://github.com/google/meridian/blob/cb0be8...
organic_media_prior_type,contribution,verified_default,Orgânico sem gasto; não calcular ROI.,https://github.com/google/meridian/blob/cb0be8...
non_media_treatments_prior_type,contribution,verified_default,Promo pode ter contribuição com sinal.,https://github.com/google/meridian/blob/cb0be8...
non_media_baseline_values,{'Promo': 0.0},explicit_choice,Zero observado como intervenção de referência;...,https://github.com/google/meridian/blob/cb0be8...
max_lag,8,verified_default,"Janela de referência, não inferida dos lags ex...",https://github.com/google/meridian/blob/cb0be8...
hill_before_adstock,False,verified_default,Adstock antes de Hill.,https://github.com/google/meridian/blob/cb0be8...
unique_sigma_for_each_geo,False,verified_default,Referência parcimoniosa na escala transformada.,https://github.com/google/meridian/blob/cb0be8...
knots,None,verified_default,156 knots GEO; nacional teria 1. Comparação fu...,https://github.com/google/meridian/blob/cb0be8...
enable_aks,False,verified_default,Nenhuma seleção automática executada.,https://github.com/google/meridian/blob/cb0be8...


,shape,dims
kpi,"[40, 156]","[geo, time]"
population,[40],[geo]
media,"[40, 156, 5]","[geo, media_time, media_channel]"
media_spend,"[40, 156, 5]","[geo, time, media_channel]"
controls,"[40, 156, 2]","[geo, time, control_variable]"
organic_media,"[40, 156, 1]","[geo, media_time, organic_media_channel]"
non_media_treatments,"[40, 156, 1]","[geo, time, non_media_channel]"


/root/.local/lib/python3.12/site-packages/meridian/data/input_data.py:605: UserWarning: Consider setting custom priors, as kpi_type was specified as `non_revenue` with no `revenue_per_kpi` being set. Otherwise, the total media contribution prior will be used with `p_mean=0.4` and `p_sd=0.2`. Further documentation available at https://developers.google.com/meridian/docs/advanced-modeling/unknown-revenue-kpi-custom#set-total-paid-media-contribution-prior
  warnings.warn(
ModelSpec construído: ModelSpec
Tipo de prior paga: contribution


## 6. As priors que o candidato carrega
Prior é a distribuição anterior à atualização; likelihood é a contribuição probabilística dos dados; posterior é a atualização. Criar esses objetos não é estimar a posterior.

A prior de contribuição paga e orgânica Beta(1,99) tem média de 1% por canal. Isso é uma hipótese forte, sem evidência de negócio nesta base: **REVIEW**. Priors de controles admitem sinal; Promo usa baseline 0 e prior de contribuição com sinal. Log-normal de efeitos de mídia impõe positividade. A Entrega 2 deve examinar compatibilidade preditiva, escala, baseline e contribuições antes de amostrar posterior.

O aviso genérico de InputData sobre prior total de mídia 0,4/0,2 não descreve automaticamente nossa prior ativa: o candidato escolhe explicitamente `contribution`, não ROI default. A distinção foi conferida no código da versão. Parâmetros RF inativos podem aparecer no registro default e não significam que reach/frequency tenham sido fornecidos.

In [4]:
priors = inspect_priors(model_spec)
active = ["contribution_m","contribution_om","contribution_n","gamma_c","xi_c","xi_n",
          "tau_g_excl_baseline","knot_values","sigma","eta_m","eta_om","alpha_m","alpha_om","ec_m","ec_om","slope_m","slope_om"]
display(pd.DataFrame({k:priors[k] for k in active}).T)
print("Nenhuma amostra prior ou posterior foi gerada.")

,distribution,parameters
contribution_m,Beta,"{'concentration1': 1.0, 'concentration0': 99.0}"
contribution_om,Beta,"{'concentration1': 1.0, 'concentration0': 99.0}"
contribution_n,TruncatedNormal,"{'loc': 0.0, 'scale': 0.1, 'low': -1.0, 'high'..."
gamma_c,Normal,"{'loc': 0.0, 'scale': 5.0}"
xi_c,HalfNormal,{'scale': 5.0}
xi_n,HalfNormal,{'scale': 5.0}
tau_g_excl_baseline,Normal,"{'loc': 0.0, 'scale': 5.0}"
knot_values,Normal,"{'loc': 0.0, 'scale': 5.0}"
sigma,HalfNormal,{'scale': 5.0}
eta_m,HalfNormal,{'scale': 1.0}


Nenhuma amostra prior ou posterior foi gerada.


## 7. Holdout GEO: materializar, alinhar e auditar
A reserva usa somente chaves GEO/tempo e seed fixa. Seis mercados por semana, distribuídos em ciclo aleatorizado, dão 15% das células; cada GEO fica com 23 ou 24 semanas. A mídia permanece para representar persistência mesmo quando o KPI é excluído da likelihood.

A atividade de Channel2 importa: um conjunto sem exposição não avaliaria seu comportamento. O teste exige atividade em cada GEO nos dois subconjuntos. A máscara não é escolhida pelo KPI, nem ressorteada para melhorar resultados. O CSV efetivo é persistido e confrontado com o objeto real do Meridian.

In [5]:
from src.validation_design import make_holdout, coverage, aligned_mask
vc = yaml.safe_load((root/"configs/validation_design.yaml").read_text())["H_GEO"]
split = make_holdout(df[["geo","time"]],seed=vc["seed"],per_week=vc["per_week"])
stored_split = pd.read_csv(root/"data/splits/holdout_geo.csv")
pd.testing.assert_frame_equal(split,stored_split)
coverage_tables = coverage(df,split)
for name,table in coverage_tables.items():
    print(name)
    display(table)
holdout_spec = build_spec(input_data,split)
from meridian.model import model
mmm = model.Meridian(input_data=input_data,model_spec=holdout_spec)
np.testing.assert_array_equal(np.asarray(mmm.holdout_id), aligned_mask(split,input_data.geo.values,input_data.time.values))
print("Holdout efetivo:", int(np.asarray(mmm.holdout_id).sum()), "células. Sem amostragem.")

,geo,holdout_n,total_n,holdout_fraction
0,Geo0,23,156,0.147436
1,Geo1,24,156,0.153846
2,Geo10,23,156,0.147436
3,Geo11,23,156,0.147436
4,Geo12,23,156,0.147436
5,Geo13,23,156,0.147436
6,Geo14,24,156,0.153846
7,Geo15,23,156,0.147436
8,Geo16,23,156,0.147436
9,Geo17,23,156,0.147436


,time,holdout_n,total_n,holdout_fraction
0,2021-01-25,6,40,0.15
1,2021-02-01,6,40,0.15
2,2021-02-08,6,40,0.15
3,2021-02-15,6,40,0.15
4,2021-02-22,6,40,0.15
...,...,...,...,...
151,2023-12-18,6,40,0.15
152,2023-12-25,6,40,0.15
153,2024-01-01,6,40,0.15
154,2024-01-08,6,40,0.15


,channel,subset,cells,active_cells,activity_fraction,exposure,spend
0,Channel0,train,5304,4672,0.880845,4.698428e+09,3.445227e+07
1,Channel0,test,936,813,0.868590,8.249928e+08,6.049444e+06
2,Channel1,train,5304,3837,0.723416,2.761602e+09,2.662525e+07
3,Channel1,test,936,661,0.706197,4.869763e+08,4.695051e+06
4,Channel2,train,5304,1912,0.360483,1.368251e+09,1.016737e+07
5,Channel2,test,936,318,0.339744,2.573000e+08,1.911977e+06
6,Channel3,train,5304,5167,0.974170,9.566995e+09,7.455407e+07
7,Channel3,test,936,904,0.965812,1.707502e+09,1.330629e+07
8,Channel4,train,5304,4621,0.871229,5.205848e+09,4.056350e+07
9,Channel4,test,936,799,0.853632,9.198230e+08,7.167180e+06


,geo,subset,cells,channel2_active
0,Geo0,train,133,49
1,Geo0,test,23,5
2,Geo1,train,132,49
3,Geo1,test,24,10
4,Geo10,train,133,47
...,...,...,...,...
75,Geo7,test,24,9
76,Geo8,train,132,43
77,Geo8,test,24,4
78,Geo9,train,133,44


geo
time
channels
channel2_geo
Holdout efetivo: 936 células. Sem amostragem.


**Interpretação:** essa reserva avalia reconstrução geotemporal condicionada à mídia disponível. Não é previsão do futuro nem experimento aleatório de exposição. Ainda se deve auditar se transformações e priors usam escala global do KPI; não foi demonstrada ausência de todo uso indireto do teste.

## 8. Nacional e experimentos futuros
H-COMP permanece `PENDING_METHOD_DECISION`: usar total nacional contendo KPIs reservados no GEO produz vazamento; reservar semanas inteiras deixa knots semanais sem informação local. Uma proposta futura usa semanas não contíguas compartilhadas e estrutura temporal previamente justificada. Se não for defensável, separar validação preditiva de comparação inferencial.

Somar conversões, mídia e gasto; contar população uma vez por GEO; receita por conversão ponderada por conversões reconcilia receita. Controles e Promo exigem hipóteses sobre índices/intensidades, não soma automática. Não construímos nem treinamos modelo nacional.

R² deve usar o mesmo alvo nacional, após agregar previsões GEO. Estimandos devem ter horizonte, unidade e intervenção equivalentes. Diferenças entre knots, priors e transformações não serão atribuídas automaticamente à informação GEO.

In [6]:
registry = yaml.safe_load((root/"configs/experiment_registry.yaml").read_text())
experiments = pd.DataFrame(registry["experiments"])
assert experiments.status.eq("planned").all()
display(experiments[["id","future_delivery","hypothesis","changed_factor","status"]])
print("Gates:", " → ".join(registry["gates"]))

,id,future_delivery,hypothesis,changed_factor,status
0,G0,2,Configuração GEO de referência,nenhum,planned
1,G1,2,Modelo sustenta células reservadas,holdout H-GEO,planned
2,N0,4,Agregação altera informação inferencial,estrutura nacional e agregação documentadas,planned
3,P1,3,Defaults informam pouco ou muito os efeitos,referência de priors,planned
4,P2,3,Menor concentração altera inferência,concentração Beta dividida por 2 com média man...,planned
5,P3,3,Maior concentração domina os dados,concentração Beta multiplicada por 2 com média...,planned
6,P4,3,Parametrização modifica conclusões,"ROI versus contribuição, somente com unidades ...",planned
7,A1,3,Janela de referência é suficiente,max_lag=8 referência,planned
8,A2,3,Janela curta altera efeitos,max_lag=4,planned
9,A3,3,Janela longa altera efeitos,max_lag=12,planned


Gates: data_ready → spec_ready → sampling_healthy → fit_reviewed → inference_reviewed → optimization_eligible


## 9. Evidências de prontidão e limites
R-hat ≤1,01, ESS bulk/tail ≥400, traces, divergências disponíveis, PPC, baseline e sensibilidade são critérios futuros, não métricas existentes. R² alto não valida causalidade. Os gates separam dados válidos, especificação revisada, amostragem saudável, ajuste e inferência antes de otimização.

O pipeline abaixo regenera auditorias e confere a máscara existente, sem substituir silenciosamente um split diferente. Os testes incluem entradas inválidas e construção real da API. PASS significa evidência técnica verificada; REVIEW indica decisão científica pendente; BLOCKED indica impossibilidade técnica.

In [7]:
from src.run_design import main
manifest = main()
subprocess.run([sys.executable,"-m","pytest","-q"],check=True)
display(pd.DataFrame(manifest["result"].items(),columns=["Item","Resultado"]))
print("Pronto para auditoria da Entrega 1; não inicia a Entrega 2.")

,Item,Resultado
0,data_ready,PASS
1,holdout_coverage,PASS
2,api,PASS
3,spec_scientific,REVIEW
4,H_COMP,PENDING_METHOD_DECISION
5,hosted_colab,NOT_TESTED_IN_HOSTED_COLAB
6,posterior_sampled,False
7,prior_sampled,False
8,optimization_executed,False


/root/.local/lib/python3.12/site-packages/meridian/data/input_data.py:605: UserWarning: Consider setting custom priors, as kpi_type was specified as `non_revenue` with no `revenue_per_kpi` being set. Otherwise, the total media contribution prior will be used with `p_mean=0.4` and `p_sd=0.2`. Further documentation available at https://developers.google.com/meridian/docs/advanced-modeling/unknown-revenue-kpi-custom#set-total-paid-media-contribution-prior
  warnings.warn(
{
  "data_ready": "PASS",
  "holdout_coverage": "PASS",
  "api": "PASS",
  "spec_scientific": "REVIEW",
  "H_COMP": "PENDING_METHOD_DECISION",
  "hosted_colab": "NOT_TESTED_IN_HOSTED_COLAB",
  "posterior_sampled": false,
  "prior_sampled": false,
  "optimization_executed": false
}
Pronto para auditoria da Entrega 1; não inicia a Entrega 2.


**Pendências antes da posterior:** revisar priors de contribuição, hipótese dos controles e Promo, efeito de borda sem pré-histórico e dependência das transformações em relação ao KPI reservado. H-COMP e agregação de índices são decisões necessárias antes da Entrega 4. Unidade monetária é necessária antes de interpretar ROI monetário.

**O que não foi feito:** amostragem de prior ou posterior, MCMC, ROI/mROI estimados, otimização, sensibilidade ou modelo nacional. A execução local não comprova uma sessão hospedada no Colab. Consulte o capítulo 11 do guia para registrar essa validação futura.